# 02.1 Raw Data Analysis

Inspect raw and paired datasets end-to-end before training.

This notebook is for **sanity checks**:
- raw `.h5ad` overview (cells, genes, key metadata)
- paired tokenized datasets (`src` / `tgt`) overview
- frozen split inspection (train/val/test)
- optional tokenization-pool check (`223k`-style full pool)

It is read-only analysis by default; no retraining is triggered.

In [8]:
from __future__ import annotations

from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
import anndata as ad
from datasets import load_from_disk
from IPython.display import display

WORKSPACE = Path('/home/stuke1/perturbgen')
REPO_ROOT = WORKSPACE / 'Perturbgen'
TOKENIZED_ROOT = WORKSPACE / 'T_perturb' / 'tokenized_data' / 'LPS_all_tps_2k'

RAW_H5AD_PATH = REPO_ROOT / 'docs/examples/lps_otar.h5ad'
SRC_DIR = TOKENIZED_ROOT / 'dataset_2000_hvg_src'
TGT_DIR = TOKENIZED_ROOT / 'dataset_2000_hvg_tgt'
SRC_H5AD_DIR = TOKENIZED_ROOT / 'h5ad_pairing_2000_hvg_src'
TGT_H5AD_DIR = TOKENIZED_ROOT / 'h5ad_pairing_2000_hvg_tgt'
SPLIT_JSON = TOKENIZED_ROOT / 'splits/stratified_cell_type_harmonized_seed42_80_10_10.json'
SPLIT_PKL = TOKENIZED_ROOT / 'splits/stratified_cell_type_harmonized_seed42_80_10_10.pkl'

# Optional: inspect the pre-pair tokenized full pool (~223k rows)
FULL_POOL_DATASET = TOKENIZED_ROOT / 'dataset_2000_hvg/LPS_all_tps_2k.dataset'

print('RAW_H5AD_PATH:', RAW_H5AD_PATH)
print('TOKENIZED_ROOT:', TOKENIZED_ROOT)
print('SPLIT_PKL exists:', SPLIT_PKL.exists())

RAW_H5AD_PATH: /home/stuke1/perturbgen/Perturbgen/docs/examples/lps_otar.h5ad
TOKENIZED_ROOT: /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k
SPLIT_PKL exists: True


## 1) Raw `.h5ad` overview

This checks the original matrix before tokenization/pairing.

In [2]:
adata_raw = ad.read_h5ad(RAW_H5AD_PATH)
print('raw shape (n_cells, n_genes):', adata_raw.shape)

key_obs = [c for c in ['time_after_LPS', 'cell_type_harmonized', 'batch', 'patient_id'] if c in adata_raw.obs.columns]
print('key obs columns present:', key_obs)

for col in key_obs:
    vc = adata_raw.obs[col].astype(str).value_counts(dropna=False)
    print(f'\n[{col}] unique={vc.shape[0]}')
    display(vc.head(12).to_frame('count'))

raw shape (n_cells, n_genes): (223478, 13826)
key obs columns present: ['time_after_LPS', 'cell_type_harmonized', 'batch', 'patient_id']

[time_after_LPS] unique=4


,count
time_after_LPS,
normal,148107
6h_LPS,43803
10h_LPS,20883
90m_LPS,10685



[cell_type_harmonized] unique=15


,count
cell_type_harmonized,
CD4+ T cells,57329
CD14 monocytes,45792
CD8+ T cells,43532
NK,21662
B cell,21477
gamma-delta T cell,8273
mucosal invariant T cell,6042
CD16 monocytes,4418
Dendritic cells,3651



[batch] unique=2


,count
batch,
0,119337
1,104141



[patient_id] unique=58


,count
patient_id,
IVLPS04_Baseline,14223
MH8919333,12028
IVLPS06_6h,10927
IVLPS03_6h,10817
IVLPS02_Baseline,10484
IVLPS02_6h,10439
IVLPS06_Baseline,10067
IVLPS03_Baseline,8640
IVLPS01_Baseline,8403


## 2) Tokenized paired datasets (`src` / `tgt`)

This is the main downstream universe used by training/evaluation notebooks.

In [3]:
def length_stats(input_ids_col):
    lens = np.asarray([len(x) for x in input_ids_col], dtype=np.int32)
    return {
        'len_min': int(lens.min()),
        'len_mean': float(lens.mean()),
        'len_median': float(np.median(lens)),
        'len_max': int(lens.max()),
    }

src_paths = sorted(SRC_DIR.glob('*.dataset'))
tgt_paths = sorted(TGT_DIR.glob('*.dataset'))

if not src_paths:
    raise FileNotFoundError(f'No src dataset found under {SRC_DIR}')
if not tgt_paths:
    raise FileNotFoundError(f'No tgt datasets found under {TGT_DIR}')

src_ds = {p.stem: load_from_disk(str(p)) for p in src_paths}
tgt_ds = {p.stem: load_from_disk(str(p)) for p in tgt_paths}

rows = []
for name, ds in src_ds.items():
    rows.append({
        'split': 'src',
        'name': name,
        'n_rows': len(ds),
        'n_columns': len(ds.column_names),
        **length_stats(ds['input_ids'][: min(10000, len(ds))]),
    })
for name, ds in tgt_ds.items():
    rows.append({
        'split': 'tgt',
        'name': name,
        'n_rows': len(ds),
        'n_columns': len(ds.column_names),
        **length_stats(ds['input_ids'][: min(10000, len(ds))]),
    })

summary_df = pd.DataFrame(rows).sort_values(['split', 'name']).reset_index(drop=True)
display(summary_df)

print('\nColumns in src datasets:')
for name, ds in src_ds.items():
    print(f'- {name}: {ds.column_names}')
print('\nColumns in tgt datasets:')
for name, ds in tgt_ds.items():
    print(f'- {name}: {ds.column_names}')

,split,name,n_rows,n_columns,len_min,len_mean,len_median,len_max
0,src,normal,148107,5,34,141.8701,135.0,452
1,tgt,1_90m_LPS,148107,5,29,131.3553,129.0,295
2,tgt,2_6h_LPS,148107,5,38,129.8504,124.0,399
3,tgt,3_10h_LPS,148107,5,27,116.9504,112.0,285



Columns in src datasets:
- normal: ['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']

Columns in tgt datasets:
- 1_90m_LPS: ['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']
- 2_6h_LPS: ['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']
- 3_10h_LPS: ['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']


## 3) Frozen split inspection (train / val / test)

This is the exact split artifact used downstream.

In [4]:
if not SPLIT_PKL.exists():
    raise FileNotFoundError(SPLIT_PKL)

with open(SPLIT_PKL, 'rb') as f:
    split = pickle.load(f)

train_idx = np.asarray(split['train_indices'])
val_idx = np.asarray(split['val_indices'])
test_idx = np.asarray(split['test_indices'])

counts_df = pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'n': [train_idx.size, val_idx.size, test_idx.size],
    'fraction': [train_idx.size, val_idx.size, test_idx.size] / np.array([train_idx.size + val_idx.size + test_idx.size] * 3),
})
print('Split metadata:')
if SPLIT_JSON.exists():
    split_meta = json.loads(SPLIT_JSON.read_text())
    display(pd.Series(split_meta)[['dataset_name', 'n_rows', 'split_obs', 'train_prop', 'test_prop', 'seed']])

display(counts_df)

# Show test-label composition on first target dataset
first_tgt_name = sorted(tgt_ds.keys())[0]
first_tgt = tgt_ds[first_tgt_name]
print(f'First target for inspection: {first_tgt_name}')

if 'cell_type_harmonized' in first_tgt.column_names:
    vc = pd.Series(first_tgt.select(test_idx.tolist())['cell_type_harmonized']).value_counts().head(15)
    display(vc.to_frame('test_count_top15'))

if 'time_after_LPS' in first_tgt.column_names:
    tvc = pd.Series(first_tgt.select(test_idx.tolist())['time_after_LPS']).value_counts()
    display(tvc.to_frame('test_time_counts'))

Split metadata:


dataset_name            LPS_all_tps_2k
n_rows                          148107
split_obs       [cell_type_harmonized]
train_prop                         0.8
test_prop                          0.1
seed                                42
dtype: object

,split,n,fraction
0,train,118485,0.799996
1,val,14811,0.100002
2,test,14811,0.100002


First target for inspection: 1_90m_LPS


,test_count_top15
CD4+ T cells,4226
CD8+ T cells,3023
CD14 monocytes,2136
NK,1799
B cell,1086
gamma-delta T cell,695
mucosal invariant T cell,495
CD16 monocytes,408
Dendritic cells,275
NKT,212


,test_time_counts
90m_LPS,14811


## 4) Optional: full tokenization-pool check (`~223k`)

Use this to reconcile the "full dataset" size vs paired/split sizes.

In [5]:
if FULL_POOL_DATASET.exists():
    full_ds = load_from_disk(str(FULL_POOL_DATASET))
    print('Full tokenized pool rows:', len(full_ds))
    print('Full tokenized pool columns:', full_ds.column_names)

    if 'time_after_LPS' in full_ds.column_names:
        full_time = pd.Series(full_ds['time_after_LPS']).value_counts()
        display(full_time.to_frame('full_pool_time_counts'))
else:
    print('Full pool dataset not found:', FULL_POOL_DATASET)

Full tokenized pool rows: 223478
Full tokenized pool columns: ['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']


,full_pool_time_counts
normal,148107
6h_LPS,43803
10h_LPS,20883
90m_LPS,10685


## 5) Optional: concrete source-target pairing examples

Shows a few shared `cell_pairing_index` rows between one source and each target dataset.

## 6) Row/feature changes step-by-step (why counts drop)

This cell gives one compact table from raw data to Notebook 4 embedding output.

In [9]:
# Paths for cross-notebook artifact checks
PAIRED_SRC_H5AD = TOKENIZED_ROOT / 'h5ad_pairing_2000_hvg_src/normal.h5ad'
PAIRED_TGT_H5AD = TOKENIZED_ROOT / 'h5ad_pairing_2000_hvg_tgt/1_90m_LPS.h5ad'
EMB_H5AD = Path('/mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/masking_split/embeddings/20260809-11:31_inference_embs_t[1, 2, 3]_scmaskgit_mpow.h5ad')

paired_src = ad.read_h5ad(PAIRED_SRC_H5AD)
paired_tgt1 = ad.read_h5ad(PAIRED_TGT_H5AD)

with open(SPLIT_PKL, 'rb') as f:
    split = pickle.load(f)

emb = ad.read_h5ad(EMB_H5AD) if EMB_H5AD.exists() else None

rows = [
    {
        'stage': 'Raw h5ad (before HVG/tokenization)',
        'rows_cells': int(adata_raw.n_obs),
        'features': int(adata_raw.n_vars),
        'note': 'All cells and genes in raw file',
    },
    {
        'stage': 'Paired h5ad (HVG 2k)',
        'rows_cells': int(paired_src.n_obs),
        'features': int(paired_src.n_vars),
        'note': 'HVG filter + stratified pairing universe',
    },
    {
        'stage': 'Tokenized full pool (.dataset)',
        'rows_cells': int(len(load_from_disk(str(FULL_POOL_DATASET)))) if FULL_POOL_DATASET.exists() else np.nan,
        'features': np.nan,
        'note': '223k pool before paired src/tgt resampling',
    },
    {
        'stage': 'Tokenized paired src (.dataset)',
        'rows_cells': int(len(src_ds[sorted(src_ds.keys())[0]])),
        'features': np.nan,
        'note': 'Rows match paired h5ad; tokens are variable-length',
    },
    {
        'stage': 'Frozen split: train',
        'rows_cells': int(len(split['train_indices'])),
        'features': np.nan,
        'note': 'Stratified split indices',
    },
    {
        'stage': 'Frozen split: val',
        'rows_cells': int(len(split['val_indices'])),
        'features': np.nan,
        'note': 'Stratified split indices',
    },
    {
        'stage': 'Frozen split: test',
        'rows_cells': int(len(split['test_indices'])),
        'features': np.nan,
        'note': 'Stratified split indices',
    },
]

if emb is not None:
    rows.append(
        {
            'stage': 'Notebook 4 embedding output (.h5ad)',
            'rows_cells': int(emb.n_obs),
            'features': int(emb.n_vars),
            'note': 'Export artifact (not equal to split size by design)',
        }
    )

flow_df = pd.DataFrame(rows)
flow_df['rows_vs_raw_pct'] = (flow_df['rows_cells'] / adata_raw.n_obs * 100).round(2)
display(flow_df)

print('\nImportant:')
print('- HVG 2k appears at paired h5ad stage as n_vars=2000.')
print('- Tokenized datasets store variable-length token sequences, not fixed 2000-length vectors.')
print('- Split reduces rows to test=14811 when evaluating held-out only.')
print('- Notebook 4 export row count can differ from split size depending on extraction/aggregation path.')

/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


,stage,rows_cells,features,note,rows_vs_raw_pct
0,Raw h5ad (before HVG/tokenization),223478,13826.0,All cells and genes in raw file,100.00
1,Paired h5ad (HVG 2k),148107,2000.0,HVG filter + stratified pairing universe,66.27
2,Tokenized full pool (.dataset),223478,NaN,223k pool before paired src/tgt resampling,100.00
3,Tokenized paired src (.dataset),148107,NaN,Rows match paired h5ad; tokens are variable-le...,66.27
4,Frozen split: train,118485,NaN,Stratified split indices,53.02
5,Frozen split: val,14811,NaN,Stratified split indices,6.63
6,Frozen split: test,14811,NaN,Stratified split indices,6.63
7,Notebook 4 embedding output (.h5ad),26856,1856.0,Export artifact (not equal to split size by de...,12.02



Important:
- HVG 2k appears at paired h5ad stage as n_vars=2000.
- Tokenized datasets store variable-length token sequences, not fixed 2000-length vectors.
- Split reduces rows to test=14811 when evaluating held-out only.
- Notebook 4 export row count can differ from split size depending on extraction/aggregation path.


In [10]:
import random

src_name = sorted(src_ds.keys())[0]
s = src_ds[src_name]

if 'cell_pairing_index' not in s.column_names:
    print('No cell_pairing_index in src; cannot show pair examples.')
else:
    src_lookup = {}
    for i, pid in enumerate(s['cell_pairing_index']):
        src_lookup.setdefault(int(pid), i)

    rng = random.Random(42)
    examples = []

    for tgt_name, ds in tgt_ds.items():
        if 'cell_pairing_index' not in ds.column_names:
            continue
        tgt_ids = [int(x) for x in ds['cell_pairing_index']]
        shared = [pid for pid in tgt_ids if pid in src_lookup]
        if not shared:
            continue

        sample_ids = rng.sample(shared, min(4, len(shared)))
        for pid in sample_ids:
            srow = s[src_lookup[pid]]
            tidx = next(i for i, x in enumerate(tgt_ids) if x == pid)
            trow = ds[tidx]
            examples.append({
                'src': src_name,
                'tgt': tgt_name,
                'pair_id': pid,
                'src_time': srow.get('time_after_LPS', None),
                'tgt_time': trow.get('time_after_LPS', None),
                'src_cell_type': srow.get('cell_type_harmonized', None),
                'tgt_cell_type': trow.get('cell_type_harmonized', None),
                'src_len': len(srow['input_ids']),
                'tgt_len': len(trow['input_ids']),
            })

    display(pd.DataFrame(examples))

""


In [13]:
examples

[]